# RetinaScan AI — Baseline CNN Model

**Status: Already completed on Google Colab (Feb 2026)**

Results achieved:
- Architecture : Simple 3-layer CNN
- Epochs run   : 4 (out of 15, due to time constraints)
- Val Accuracy : ~50%
- Purpose      : Prove end-to-end pipeline works before transfer learning

This notebook is kept for reference and documentation.
Phase 2 uses EfficientNetB3 and ResNet50 (notebooks 04 and 05) for 80%+ accuracy.

## Baseline Results Summary

| Metric | Value |
|--------|-------|
| Model | Simple CNN (3 conv blocks) |
| Dataset | APTOS 2019 (3662 images) |
| Train/Val Split | 80/20 stratified |
| Epochs | 4 |
| Val Accuracy | ~50% |
| Purpose | Baseline benchmark |

## Cell 1: Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, accuracy_score
)

sys.path.append('../src')

np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## Cell 2: Load Data

In [ ]:
# Load CSV
train_csv = pd.read_csv('../data/raw/train.csv')

# Use preprocessed images
train_csv['image_path'] = train_csv['id_code'].apply(
    lambda x: f'../data/processed/train_images/{x}.png'
)

# Filter existing
train_csv = train_csv[train_csv['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(f'Total images: {len(train_csv)}')
print(train_csv['diagnosis'].value_counts().sort_index())

## Cell 3: Train/Val Split

In [ ]:
train_df, val_df = train_test_split(
    train_csv,
    test_size=0.20,
    stratify=train_csv['diagnosis'],
    random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)}')

## Cell 4: Class Weights

In [ ]:
classes = np.unique(train_df['diagnosis'])
weights = compute_class_weight('balanced', classes=classes, y=train_df['diagnosis'])
class_weights = dict(zip(classes, weights))

print('Class weights:')
for k, v in class_weights.items():
    print(f'  Grade {k}: {v:.4f}')

## Cell 5: Data Generator

In [ ]:
def create_generator(dataframe, batch_size, img_size=224, shuffle=True, augment=False):
    num_samples = len(dataframe)
    num_classes = 5

    while True:
        if shuffle:
            dataframe = dataframe.sample(frac=1).reset_index(drop=True)

        for offset in range(0, num_samples, batch_size):
            batch_df = dataframe.iloc[offset:offset + batch_size]
            batch_images, batch_labels = [], []

            for _, row in batch_df.iterrows():
                try:
                    img = cv2.imread(row['image_path'])
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, (img_size, img_size))
                    img = img.astype(np.float32) / 255.0

                    if augment:
                        if np.random.random() > 0.5:
                            img = np.fliplr(img)
                        if np.random.random() > 0.5:
                            img = np.flipud(img)

                    batch_images.append(img)
                    batch_labels.append(row['diagnosis'])
                except:
                    continue

            if len(batch_images) > 0:
                yield np.array(batch_images, dtype=np.float32), to_categorical(batch_labels, num_classes)


IMG_SIZE   = 224
BATCH_SIZE = 32

train_gen = create_generator(train_df, BATCH_SIZE, IMG_SIZE, shuffle=True,  augment=True)
val_gen   = create_generator(val_df,   BATCH_SIZE, IMG_SIZE, shuffle=False, augment=False)

steps_per_epoch  = len(train_df) // BATCH_SIZE
validation_steps = len(val_df)   // BATCH_SIZE

print(f'Steps per epoch  : {steps_per_epoch}')
print(f'Validation steps : {validation_steps}')

## Cell 6: Build Baseline CNN

In [ ]:
def build_baseline_cnn(input_shape=(224, 224, 3), num_classes=5):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model


model = build_baseline_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)
model.summary()

## Cell 7: Train

**Note:** This was originally run on Google Colab with T4 GPU.
Running locally without GPU will be very slow.

In [ ]:
os.makedirs('../models', exist_ok=True)

callback_list = [
    callbacks.ModelCheckpoint(
        '../models/baseline_cnn_best.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    )
]

# NOTE: Originally ran 4 epochs on Colab → 50% val accuracy
history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=15,
    validation_data=val_gen,
    validation_steps=validation_steps,
    callbacks=callback_list,
    verbose=1
)

## Cell 8: Plot Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Baseline CNN — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Baseline CNN — Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../models/baseline_history.png', dpi=300, bbox_inches='tight')
plt.show()